# GECCO Unconstrained Cache Generation: Ultra-VETime Dual-Subsystem Teacher
### High-Density Curriculum Caching & Dedicated Head-to-Head Evaluation Cache
- **Domain**: GECCO 2018 Industrial Challenge (Drinking Water Quality Telemetry, 139,566 steps, 9 channels)
- **Backbone**: `facebook/dinov2-large` (1024-dim self-supervised visual patch features)
- **Dual-Subsystem 4-Panel Dashboard**: Chemical Disinfection (ORP/pH/Cl) + Physical Quality (Leit/Trueb) on dedicated scales
- **Multi-Regime Operational Centroids**: $K=5$ KMeans nominal operating clusters from 16,000-step baseline
- **Rendering**: Fast direct Agg canvas buffer copy (zero PNG compression bottleneck)
- **Multi-GPU**: Shards candidate windows across `cuda:0` and `cuda:1` if dual GPUs are provisioned

In [ ]:
# --- Step 1: Workspace Setup & Codebase Unpacking ---
import os, sys, shutil, subprocess, base64, io, zipfile

repo_dir = '/kaggle/working/VisTAD'
os.makedirs(repo_dir, exist_ok=True)
os.chdir(repo_dir)

print('Unpacking self-contained VisTAD codebase (src, vlm4ts, configs, data)...')
b64_bundle = "..."
raw_zip = base64.b64decode(b64_bundle)
with zipfile.ZipFile(io.BytesIO(raw_zip), 'r') as z:
    z.extractall(repo_dir)
print('VisTAD codebase unpacked successfully.')
print('Extracted top-level dirs:', os.listdir(repo_dir))


In [ ]:
# --- Step 2: GPU Accelerator & Multi-GPU Diagnostics ---
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
num_gpus = torch.cuda.device_count()
print(f'Detected {num_gpus} GPU(s):')
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)')


In [ ]:
# --- Step 3: GECCO Dataset Auto-Discovery ---
import os, glob, shutil
os.makedirs('data/gecco', exist_ok=True)
gecco_target = 'data/gecco/gecco_water.csv'

candidates = glob.glob('/kaggle/input/**/gecco_water.csv', recursive=True)
if not candidates:
    candidates = glob.glob('/kaggle/input/**/gecco/*.csv', recursive=True)
if candidates:
    print(f'Found GECCO dataset candidate: {candidates[0]}')
    if not os.path.exists(gecco_target):
        shutil.copy2(candidates[0], gecco_target)
else:
    print('WARNING: Could not find gecco_water.csv in /kaggle/input!')
print(f'GECCO data file verified at {gecco_target}: {os.path.getsize(gecco_target)/(1024**2):.2f} MB')


In [ ]:
# --- Step 4: Execute GECCO Unconstrained Cache Generation ---
import os, subprocess, time
os.environ['PYTHONPATH'] = '.'

cmd = [
    'python', 'vlm4ts/generate_gecco_cache.py',
    '--data_path', 'data/gecco/gecco_water.csv',
    '--output_train_cache', 'cache/gecco_train_ultravetime_cache.pt',
    '--output_eval_cache', 'cache/gecco_eval_head_to_head_cache.pt',
    '--window_size', '200',
    '--num_train_windows', '3500',
    '--num_eval_windows', '1500',
    '--model_id', 'facebook/dinov2-large',
    '--n_regimes', '5'
]
print('Executing unconstrained caching command:', ' '.join(cmd))
t0 = time.time()
proc = subprocess.run(cmd, check=True)
duration = time.time() - t0
print(f'Caching completed successfully in {duration:.1f}s ({duration/60:.2f} min).')


In [ ]:
# --- Step 5: Verify Cache Statistics & Teacher Baseline Evaluation ---
import torch, os
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

for cache_path, label in [
    ('cache/gecco_train_ultravetime_cache.pt', 'TRAINING CURRICULUM CACHE'),
    ('cache/gecco_eval_head_to_head_cache.pt', 'HEAD-TO-HEAD EVALUATION CACHE')
]:
    if not os.path.exists(cache_path):
        print(f'WARNING: {cache_path} not found!')
        continue
    c = torch.load(cache_path, weights_only=False)
    wins = c['windows']
    meta = c.get('metadata', {})
    y_flags = [wins[k]['y_flag'] for k in wins]
    t_scores = [wins[k]['teacher_score'] for k in wins]
    feat_dim = wins[list(wins.keys())[0]]['z_teacher'].shape
    
    print(f'=== {label} ===')
    print(f'  Path:             {cache_path} ({os.path.getsize(cache_path)/(1024**2):.2f} MB)')
    print(f'  Teacher:          {meta.get("teacher_name")} ({meta.get("backbone")})')
    print(f'  Total Windows:    {len(wins)}')
    print(f'  Anomalous:        {sum(y_flags)} ({100*sum(y_flags)/len(wins):.1f}%)')
    print(f'  Nominal:          {len(wins) - sum(y_flags)} ({100*(len(wins)-sum(y_flags))/len(wins):.1f}%)')
    print(f'  Feature Dim:      {feat_dim}')
    
    try:
        auroc = roc_auc_score(y_flags, t_scores)
        prauc = average_precision_score(y_flags, t_scores)
        p, r, th = precision_recall_curve(y_flags, t_scores)
        f1s = 2 * p * r / np.clip(p + r, 1e-12, None)
        best_f1 = np.max(f1s)
        best_th = th[np.argmax(f1s)]
        print(f'  Teacher Baseline Performance on {label}:')
        print(f'    AUROC:          {auroc:.4f}')
        print(f'    PR-AUC:         {prauc:.4f}')
        print(f'    Best F1:        {best_f1:.4f} (at threshold {best_th:.4f})')
    except Exception as e:
        print(f'    Metrics computation note: {e}')
    print()


In [ ]:
# --- Step 6: Stage Outputs for Direct Kaggle Download ---
import shutil
output_dir = '/kaggle/working'
for cp in ['cache/gecco_train_ultravetime_cache.pt', 'cache/gecco_eval_head_to_head_cache.pt']:
    if os.path.exists(cp):
        target = os.path.join(output_dir, os.path.basename(cp))
        print(f'Copying {cp} -> {target}')
        shutil.copy2(cp, target)
print('All cache artifacts staged in /kaggle/working for output retrieval.')
